In [1]:
from src.fitter.analysis.juno_syst import JunoSyst
from src.fitter.tool.profile_analysis import GetProfile
from src.fitter.config import GlobalConfig as gcfg
from loguru import logger

# gcfg.dmsq31_NO = 2.6e-3 # for test if dmsq31_NO in GlobalConfig is shared in all modules
import numpy as np
import torch
import time

""" JUNO + DYB """
print("==============================JUNO + DYB fitting==============================")
gcfg.combinedyb = True
gcfg.IfJustDYB = False
gcfg.dybFineBin = True
gcfg.test_statistic = 0

""" Dataset """
# gcfg.dyb_input_path = "./data/Hanzhang/Not_Public_DYB_Total_IBD_Yield_Spec_136bins.root"
# gcfg.dyb_obj_name = {
#     "ibd_yield_spec" : "Tot_IBD_Yield_Spec_136bins",
#     "response_matrix" : "Response_240bins",
#     "relative_cov" : "Relative_Cov_ShapeOnly_136bins"
# }
gcfg.dyb_input_path = "./data/Hanzhang/DYB_2021Paper_jianrun_det_resp.root"
gcfg.dyb_obj_name["ibd_yield_spec"] = "h_tot_ibd_yield"
gcfg.dyb_obj_name["response_matrix"] = "ResponseTotalRebin"
gcfg.dyb_obj_name["relative_cov"] = "Relative_Cov_ShapeOnly"
# gcfg.dyb_input_path = "./data/Hanzhang/DYB_2025Paper.root"
# gcfg.dyb_obj_name["ibd_yield_spec"] = 'h_tot_ibd_yield'
# gcfg.dyb_obj_name["response_matrix"] = 'ResponseTotal'
# gcfg.dyb_obj_name["relative_cov"] = 'Relative_Cov_All'


""" uncertainty """
if gcfg.combinetao or gcfg.combinedyb:  # 如果有TAO或者dyb，则已经是在测量IBD yield不用加IBDyield误差
    gcfg.m_sigma_ReaCorr = 0.2e-2  # mean energy per fission uncertainty
if gcfg.combinetao and not gcfg.combinedyb:  # 如果有TAO但是没有dyb提供IBD yield约束，则加上IBDyield约束
    gcfg.IfConstrainIBDyield = True

""" new bin juno """
gcfg.IfNewBin = False

""" fitting """
gcfg.IfSimulateDYB = True
seed = 0
days = 100.0
years = days / 365.25
fitter = JunoSyst(years)
obs_juno = fitter.get_obs_spectrum(seed)
obs_dyb = fitter.y_obs_dyb

from iminuit import Minuit

N_alpha_isotope_err = fitter.rea.n_isotope_err_alpha
logger.info("fit_para_init type {}", fitter.fit_para_init.dtype)
m = Minuit(fitter.chi2, fitter.fit_para_init, name=fitter.fit_para_names)

m.tol = 0.1
isotope_err_names = [f"alpha_isotope_err_{i}" for i in range(N_alpha_isotope_err)]
for name in isotope_err_names:
    m.limits[name] = (-1, np.inf)
    # m.fixed[name] = True
t3 = time.perf_counter()
result = m.migrad()
t4 = time.perf_counter()
print(f"Time used of fitting: {t4 - t3:.5f} seconds")
chisq_min = m.fval
xs = m.values
es = m.errors
print("Best fit parameters: ")
for i in range(len(xs)):
    if i < 4:
        print(f"{fitter.fit_para_names[i]}: {xs[i]:.10f} ± {es[i]:.10f}, precision: {es[i] / xs[i] * 100.0:.5f}%")
    else:
        print(f"{fitter.fit_para_names[i]}: {xs[i]:.5f} ± {es[i]:.5f}")
print(f"Minimum chi-squared: {chisq_min:.5f}")


ModuleNotFoundError: No module named 'src.fitter'

In [ ]:
import json

errors_dict = {}
for name in m.parameters:
    errors_dict[name] = round(float(m.errors[name]), 6)

    # errors_dict[name] = float(m.errors[name])
with open("parameter_errors.json", "w") as f:
    json.dump(errors_dict, f, indent=2)

print("参数误差已保存到 parameter_errors.json")
print("可以通过以下方式读取：")
print("with open('parameter_errors.json', 'r') as f:")
print("    data = json.load(f)")
print("    print(data['sinsq13'])  # 获取sinsq13的误差")


In [ ]:
import numpy as np
from src.fitter.analysis.juno_syst import JunoSyst

params_SGD = np.loadtxt("fitted_params_approximate.txt")
junosyst = JunoSyst(years)

chi2_fit_SGD = junosyst.chi2(params_SGD)
print(f"SGD fit chi2: {chi2_fit_SGD:.16e}")

params_minuit = [m.values[name] for name in m.parameters]
chi2_fit_minuit = junosyst.chi2(params_minuit)
print(f"Minuit fit chi2: {chi2_fit_minuit:.16e}")


In [ ]:
# 计算梯度值
from src.fitter.analysis.juno_syst import JunoSyst
from src.fitter.config import GlobalConfig as gcfg
from src.fitter.fitting_frame.torch_helper import gradient_step0_report

""" JUNO + DYB """
print("==============================JUNO + DYB fitting==============================")
gcfg.combinedyb = True
gcfg.IfJustDYB = False
gcfg.dybFineBin = True
gcfg.test_statistic = 0

""" Dataset """
# gcfg.dyb_input_path = "./data/Hanzhang/Not_Public_DYB_Total_IBD_Yield_Spec_136bins.root"
# gcfg.dyb_obj_name = {
#     "ibd_yield_spec" : "Tot_IBD_Yield_Spec_136bins",
#     "response_matrix" : "Response_240bins",
#     "relative_cov" : "Relative_Cov_ShapeOnly_136bins"
# }
gcfg.dyb_input_path = "./data/Hanzhang/DYB_2021Paper_jianrun_det_resp.root"
gcfg.dyb_obj_name["ibd_yield_spec"] = "h_tot_ibd_yield"
gcfg.dyb_obj_name["response_matrix"] = "ResponseTotalRebin"
gcfg.dyb_obj_name["relative_cov"] = "Relative_Cov_ShapeOnly"
# gcfg.dyb_input_path = "./data/Hanzhang/DYB_2025Paper.root"
# gcfg.dyb_obj_name["ibd_yield_spec"] = 'h_tot_ibd_yield'
# gcfg.dyb_obj_name["response_matrix"] = 'ResponseTotal'
# gcfg.dyb_obj_name["relative_cov"] = 'Relative_Cov_All'

gcfg.scale_factor_path = "./data/Xuejq/scale_factor_juno_dyb.json"
""" uncertainty """
if gcfg.combinetao or gcfg.combinedyb:  # 如果有TAO或者dyb，则已经是在测量IBD yield不用加IBDyield误差
    gcfg.m_sigma_ReaCorr = 0.2e-2  # mean energy per fission uncertainty
if gcfg.combinetao and not gcfg.combinedyb:  # 如果有TAO但是没有dyb提供IBD yield约束，则加上IBDyield约束
    gcfg.IfConstrainIBDyield = True

""" new bin juno """
gcfg.IfNewBin = False

""" fitting """
gcfg.IfSimulateDYB = True
year = 50 / 365.25
junosyst = JunoSyst(
    year,
    n_E_nu_bins=5600,
    n_E_dep_bins=5600,
    n_E_d_bins=560,
    n_E_p_bins=560,
)

junosyst.get_obs_spectrum(seed=1)
report = gradient_step0_report(junosyst, lr=1e-9, use_parameter_scaling=True, eps_rel=1e-6)